**Master 1 Monnaie Banque Finance · Analyse de données · durée : 1 h**

**Données :** l'économie américaine trimestre par trimestre, de 1959 à 2009 (jeu `macrodata` inclus dans `statsmodels`, aucun téléchargement nécessaire). Chaque **trimestre** est un individu, décrit par 9 indicateurs : croissance du PIB, de la consommation, de l'investissement, des dépenses publiques, du revenu disponible et de la masse monétaire M1, inflation, taux court (bons du Trésor à 3 mois) et taux de chômage.

**Question :** peut-on résumer l'état de l'économie en quelques indicateurs synthétiques, et que racontent-ils sur 50 ans d'histoire économique ?

| Partie | Question centrale |
|---|---|
| 1 | Pourquoi a-t-on besoin d'une ACP ici ? |
| 2 | Que contiennent les sorties d'une ACP ? |
| 3 | Que racontent les axes, les variables, les trimestres ? |
| 4 | À quoi sert l'ACP une fois faite ? |

Les cellules contenant `...` sont à compléter . Les questions suivies d'une cellule « Votre réponse » appellent une **réponse rédigée**

> Si `statsmodels` n'est pas installé : `pip install statsmodels`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pd.set_option("display.precision", 2)
d = sm.datasets.macrodata.load_pandas().data
d.index = pd.period_range(start=f"{int(d['year'][0])}Q{int(d['quarter'][0])}", periods=len(d), freq="Q")

# Taux de croissance annualisés (en %) pour les grandeurs en volume et la masse monétaire
croissance = d[["realgdp", "realcons", "realinv", "realgovt", "realdpi", "m1"]].pct_change() * 400
X = pd.concat([croissance, d[["infl", "tbilrate", "unemp"]]], axis=1).dropna()
X.columns = ["PIB", "consommation", "investissement", "dépenses publiques", "revenu disponible",
             "M1", "inflation", "taux court", "chômage"]
n, p = X.shape
print(n, "trimestres de", X.index[0], "à", X.index[-1], "|", p, "variables")
X.head()

*Toutes les variables sont en % : taux de croissance annualisés pour les six premières, inflation annualisée, taux des bons du Trésor à 3 mois, taux de chômage.*

### Informations et fonctions fournies
Les récessions américaines officielles (datées par le NBER) et la décennie de chaque trimestre servent de **variables illustratives** : elles ne participent pas au calcul de l'ACP. Exécutez la cellule.

In [ ]:
pics =    ["1960Q2", "1969Q4", "1973Q4", "1980Q1", "1981Q3", "1990Q3", "2001Q1", "2007Q4"]
creux =   ["1961Q1", "1970Q4", "1975Q1", "1980Q3", "1982Q4", "1991Q1", "2001Q4", "2009Q2"]
recession = pd.Series(False, index=X.index)
for a, b in zip(pics, creux):
    recession[(X.index > pd.Period(a, "Q")) & (X.index <= pd.Period(b, "Q"))] = True
periode = pd.Series(np.where(X.index.year < 1970, "1959-1969", (X.index.year // 10 * 10).astype(str) + "s"),
                    index=X.index)
print(recession.sum(), "trimestres de récession")

def eboulis(lam):
    k = np.arange(1, len(lam) + 1)
    plt.figure(figsize=(8, 4))
    plt.bar(k, lam, color="#2F5DA8"); plt.plot(k, lam, "o-", color="#16233A")
    plt.axhline(1, ls="--", color="#A9501A", label="Kaiser : λ = 1")
    plt.xticks(k); plt.xlabel("axe"); plt.ylabel("valeur propre λ")
    plt.title("Éboulis des valeurs propres"); plt.legend(); plt.show()

def cercle(corr, a="F1", b="F2", pct=None):
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.add_patch(plt.Circle((0, 0), 1, fill=False))
    for var in corr.index:
        x, y = corr.loc[var, a], corr.loc[var, b]
        ax.arrow(0, 0, x, y, head_width=0.03, length_includes_head=True, color="#2F5DA8")
        ax.text(x * 1.12, y * 1.12, var, fontsize=9, ha="center", va="center")
    ax.axhline(0, color="grey", lw=0.8); ax.axvline(0, color="grey", lw=0.8)
    ax.set_xlim(-1.25, 1.25); ax.set_ylim(-1.25, 1.25); ax.set_aspect("equal")
    ax.set_xlabel(a + (f" ({pct[a]:.1f} %)" if pct is not None else ""))
    ax.set_ylabel(b + (f" ({pct[b]:.1f} %)" if pct is not None else ""))
    ax.set_title("Cercle des corrélations"); plt.show()

def plan_individus(F, groupes, a=0, b=1, etiquettes=None, titre="Trimestres"):
    fig, ax = plt.subplots(figsize=(9, 7))
    for (nom, idx), m in zip(pd.Series(groupes.values).groupby(groupes.values).groups.items(), "osD^v"):
        ax.scatter(F[idx, a], F[idx, b], marker=m, alpha=0.7, label=nom)
    for i in (etiquettes or []):
        ax.annotate(str(X.index[i]), (F[i, a], F[i, b]), xytext=(6, 6), textcoords="offset points", weight="bold")
    ax.axhline(0, color="grey", lw=0.8); ax.axvline(0, color="grey", lw=0.8)
    ax.set_aspect("equal"); ax.set_xlabel(f"F{a+1}"); ax.set_ylabel(f"F{b+1}"); ax.legend()
    ax.set_title(titre); plt.show()

def serie_temporelle(valeurs, nom):
    fig, ax = plt.subplots(figsize=(11, 3.8))
    t = X.index.to_timestamp()
    ax.plot(t, valeurs, color="#16233A", lw=1.2)
    for a, b in zip(pics, creux):
        ax.axvspan(pd.Period(a, "Q").to_timestamp(), pd.Period(b, "Q").to_timestamp(), color="#A9501A", alpha=0.2)
    ax.axhline(0, color="grey", lw=0.8); ax.set_ylabel(nom)
    ax.set_title(f"{nom} au cours du temps (bandes : récessions NBER)"); plt.show()

## Partie 1 · Pourquoi une ACP ?

**Q1.** Affichez la matrice des corrélations (code fourni). Repérez deux **groupes de variables** qui évoluent ensemble. Que mesure, en langage économique, chacun de ces groupes ?

In [ ]:
R = X.corr()
plt.figure(figsize=(8, 6.5))
plt.imshow(R, cmap="RdBu_r", vmin=-1, vmax=1); plt.colorbar(label="corrélation")
plt.xticks(range(p), X.columns, rotation=60, ha="right"); plt.yticks(range(p), X.columns)
for i in range(p):
    for j in range(p):
        plt.text(j, i, f"{R.iloc[i, j]:.1f}", ha="center", va="center", fontsize=7)
plt.title("Matrice des corrélations"); plt.show()

**Votre réponse :** …

**Q2.** Un économiste veut suivre « l'état de l'économie » chaque trimestre. Pourquoi est-il peu pratique de suivre ces 9 séries une par une ? Que gagnerait-on à les résumer en deux ou trois indicateurs ?

**Votre réponse :** …

## Partie 2 · Réaliser l'ACP et comprendre ses sorties

**Q3.** Centrez-réduisez les données puis ajustez une ACP (complétez les deux lignes).

In [ ]:
# Q3
Z = ...                       # données centrées réduites
acp = ...                     # ACP ajustée sur Z
lam = acp.explained_variance_ * (n - 1) / n       # convention 1/n du cours
F = acp.transform(Z)                              # coordonnées des trimestres
pct = pd.Series(100 * lam / lam.sum(), index=[f"F{k+1}" for k in range(p)])
pd.DataFrame({"lambda": lam, "% inertie": pct, "% cumulé": pct.cumsum()})

**Q4. Le lien avec le cours (un peu de maths).** Exécutez les trois vérifications. Pour chacune, écrivez en une phrase ce qu'elle signifie concrètement.

In [ ]:
print("1) Somme des valeurs propres :", lam.sum().round(3), " / nombre de variables :", p)
print("2) Variance de F1 :", F[:, 0].var().round(3), " / λ1 :", lam[0].round(3))
print("3) Corrélation entre F1 et F2 :", np.corrcoef(F[:, 0], F[:, 1])[0, 1].round(6))

**Votre réponse :** …

**Q5.** Tracez l'éboulis. Combien d'axes retenez-vous selon le coude, selon Kaiser ? Les deux critères sont-ils aussi nets que dans le cours ? Pourquoi, à votre avis, les parts d'inertie sont-elles plus faibles qu'avec les vins ?

In [ ]:
# Q5
...

**Votre réponse :** …

## Partie 3 · Interpréter

### 3.1 Les variables : donner un sens aux axes

**Q6.** Les corrélations variables-axes sont calculées ci-dessous. Tracez le cercle des corrélations F1–F2. **Nommez F1 et F2** en langage économique, et justifiez avec les variables les plus corrélées.

In [ ]:
corr = pd.DataFrame(acp.components_.T * np.sqrt(lam), index=X.columns,
                    columns=[f"F{k+1}" for k in range(p)])
corr[["F1", "F2", "F3"]].sort_values("F1")

In [ ]:
# Q6
...

**Votre réponse :** …

**Q7.** Quelles variables sont **mal représentées** dans le plan F1–F2 (qualité = corr² sur F1 + corr² sur F2) ? Tracez le plan F1–F3 : que mesure F3 ?

In [ ]:
# Q7
qualite = ...
display(qualite.sort_values())
...

**Votre réponse :** …

### 3.2 Les trimestres : lire l'histoire économique

**Q8.** Représentez les trimestres dans le plan F1–F2, colorés par **période**. La période n'a pas servi au calcul : c'est une variable illustrative. Décrivez le chemin parcouru par l'économie américaine de décennie en décennie. Reconnaissez-vous des épisodes connus ?

In [ ]:
# Q8
...

**Votre réponse :** …

**Q9.** Même graphique, mais coloré selon que le trimestre est **en récession** ou non. Où se trouvent les récessions ? Sur quel axe se distinguent-elles ? Toutes les récessions se ressemblent-elles ?

In [ ]:
# Q9
...

**Votre réponse :** …

**Q10. Des trimestres atypiques.** Repérez les 4 trimestres les plus éloignés du centre dans le plan F1–F2 et affichez leurs valeurs centrées réduites. Pour chacun, expliquez sa position à l'aide des noms d'axes, puis reliez-le à un événement économique que vous connaissez.

In [ ]:
# Q10
dist_plan = ...
atypiques = list(np.argsort(-dist_plan)[:4])
plan_individus(F, periode, etiquettes=atypiques, titre="Trimestres atypiques")
pd.DataFrame(Z[atypiques], index=X.index[atypiques], columns=X.columns).round(1)

**Votre réponse :** …

**Q11. Le piège du centre.** Un trimestre peut sembler « normal » parce qu'il est près du centre du graphique, alors qu'il est en réalité loin de la moyenne sur des variables que le plan ne montre pas. Pour le repérer, comparez pour chaque trimestre :
- sa **distance au centre dans le plan** F1–F2 (calculée en Q10),
- sa **distance au centre réelle**, dans l'espace des 9 variables : la racine de la somme des carrés de ses valeurs centrées réduites.

Trouvez le trimestre dont le plan montre **la plus petite part** de sa distance réelle et affichez ses valeurs centrées réduites. Ce trimestre est-il « normal » ? Pour combien de trimestres le plan montre-t-il moins de la moitié de la distance réelle ? Qu'en concluez-vous ?

In [ ]:
# Q11
dist_reelle = ...                         # racine de la somme des Z² sur les 9 variables
part_visible = dist_plan / dist_reelle    # 1 = tout est visible dans le plan, 0 = rien
i_cache = int(np.argmin(part_visible))
print(X.index[i_cache], "| distance dans le plan =", dist_plan[i_cache].round(2), "| distance réelle =", dist_reelle[i_cache].round(2))
print("trimestres dont le plan montre moins de la moitié de la distance réelle :", ...)
pd.Series(Z[i_cache], index=X.columns).round(2).sort_values()

**Votre réponse :** …

## Partie 4 · Utiliser l'ACP

### 4.1 Construire un indicateur synthétique

**Q12.** Tracez le score F1 de chaque trimestre au cours du temps, avec les récessions en fond (fonction `serie_temporelle`). Faites de même pour F2. F1 pourrait-il servir d'**indicateur d'activité** ? Et F2, que suit-il sur 50 ans ?

*Pour information : le Chicago Fed National Activity Index, publié chaque mois par la Réserve fédérale de Chicago, est construit exactement ainsi : c'est la première composante principale de 85 indicateurs économiques.*

In [ ]:
# Q12
...

**Votre réponse :** …

### 4.2 Communiquer les résultats

**Q13.** Rédigez en **5 à 6 lignes** l'encadré d'une note de conjoncture destinée à un comité de direction de banque qui n'a jamais entendu parler de valeurs propres : ce qu'on a fait, combien d'information est conservée, ce que signifient les deux indicateurs, ce qu'ils disent des récessions (par exemple celle de 2008), et une limite.

**Votre réponse :** …